In [4]:
import requests, time, re
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "http://books.toscrape.com/"
RATING = {"One":1, "Two":2, "Three":3, "Four":4, "Five":5}

def fetch(url, retries=3, timeout=1):
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=timeout)
            r.raise_for_status()
            r.encoding = "utf-8"
            return r.text
        except requests.RequestException as e:
            print(f"[{attempt+1}/{retries}] {url} -> {e}")
            time.sleep(2)
    return None

In [5]:
def parse_page(html, base_url):
    soup = BeautifulSoup(html, "html.parser")

    rows = []
    for art in soup.select("article.product_pod"):
        a = art.select_one("h3 > a")
        title = a["title"].strip()
        url = urljoin(base_url, a["href"])
        price_txt = art.select_one("p.price_color").text
        price = float(re.sub(r"[^\d.]", "", price_txt))
        rating_cls = art.select_one("p.star-rating")["class"]
        rating = next((RATING[c] for c in rating_cls if c in RATING), None)
        book_id = re.search(r"_(\d+)/", url)
        book_id = int(book_id.group(1)) if book_id else None
        rows.append({
            "id": book_id,
            "title": title,
            "price": price,
            "rating": rating,
            "url": url,
        })
    return rows

def get_category(book_url, retries=3, timeout=1):
    html = fetch(book_url, retries=retries, timeout=timeout)
    if not html:
        return None
    soup = BeautifulSoup(html, "html.parser")
    crumbs = soup.select("ul.breadcrumb li a")
    # Home > Books > <Category> > <Title>
    return crumbs[2].text.strip() if len(crumbs) > 2 else None

In [6]:
all_rows = []
for page in range(1, 3):
    url = f"{BASE_URL}catalogue/page-{page}.html"
    html = fetch(url)
    if html:
        rows = parse_page(html, url)
        for row in rows:
            row["category"] = get_category(row["url"])
        all_rows.extend(rows)
        print(f"page {page}: {len(all_rows)} total")

df = pd.DataFrame(all_rows)
df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

df.head()

page 1: 20 total
page 2: 40 total


,id,title,price,rating,url,category
0,1000,A Light in the Attic,51.77,3,http://books.toscrape.com/catalogue/a-light-in...,Poetry
1,999,Tipping the Velvet,53.74,1,http://books.toscrape.com/catalogue/tipping-th...,Historical Fiction
2,998,Soumission,50.10,1,http://books.toscrape.com/catalogue/soumission...,Fiction
3,997,Sharp Objects,47.82,4,http://books.toscrape.com/catalogue/sharp-obje...,Mystery
4,996,Sapiens: A Brief History of Humankind,54.23,5,http://books.toscrape.com/catalogue/sapiens-a-...,History


In [8]:
df.to_csv("books.csv", index=False, encoding="utf-8")

Карточка книги — article.product_pod: контейнер одного товара, по нему цикл.

Название — h3 > a, атрибут title: полное имя без обрезки.

Ссылка — h3 > a, атрибут href: относительный путь, через urljoin → абсолютный.

Цена — p.price_color: текст £51.77, чистим regex.

Рейтинг — p.star-rating, атрибут class: ["star-rating", "Three"] → маппим в число.

Категория — ul.breadcrumb li a, индекс [2]: хлебные крошки, только на детальной странице.

ID — re.search(r"_(\d+)/", url): regex по URL.